In [0]:
from pyspark.sql.types import StructType, StructField, \
    StringType, IntegerType

# Skip CSV — create directly from list
data = [
    ("Ravi",   "Engineering", "Pune",      55000, "M", 28, "2021-03-15"),
    ("Priya",  "HR",          "Mumbai",    42000, "F", 32, "2019-07-22"),
    ("Arjun",  "Engineering", "Delhi",     72000, "M", 26, "2022-01-10"),
    ("Sneha",  "Finance",     "Pune",      61000, "F", 30, "2020-11-05"),
    ("Rohit",  "Engineering", "Mumbai",    80000, "M", 35, "2018-06-30"),
    ("Meera",  "HR",          "Bangalore", 39000, "F", 27, "2023-02-18"),
    ("Karan",  "Finance",     "Delhi",     55000, "M", 29, "2021-09-12"),
    ("Divya",  "Engineering", "Pune",      91000, "F", 33, "2017-04-25"),
    ("Nitin",  "HR",          "Mumbai",    44000, "M", 31, "2020-03-08"),
    ("Anjali", "Finance",     "Bangalore", 67000, "F", 28, "2019-11-30"),
]

schema = StructType([
    StructField("name",         StringType(),  True),
    StructField("dept",         StringType(),  True),
    StructField("city",         StringType(),  True),
    StructField("salary",       IntegerType(), True),
    StructField("gender",       StringType(),  True),
    StructField("age",          IntegerType(), True),
    StructField("joining_date", StringType(),  True),
])

df = spark.createDataFrame(data, schema)
display(df)
df.printSchema()
print("✅ DataFrame ready! Rows:", df.count())

name,dept,city,salary,gender,age,joining_date
Ravi,Engineering,Pune,55000,M,28,2021-03-15
Priya,HR,Mumbai,42000,F,32,2019-07-22
Arjun,Engineering,Delhi,72000,M,26,2022-01-10
Sneha,Finance,Pune,61000,F,30,2020-11-05
Rohit,Engineering,Mumbai,80000,M,35,2018-06-30
Meera,HR,Bangalore,39000,F,27,2023-02-18
Karan,Finance,Delhi,55000,M,29,2021-09-12
Divya,Engineering,Pune,91000,F,33,2017-04-25
Nitin,HR,Mumbai,44000,M,31,2020-03-08
Anjali,Finance,Bangalore,67000,F,28,2019-11-30


root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- joining_date: string (nullable = true)

✅ DataFrame ready! Rows: 10


In [0]:
# Write as Delta table — this is the Databricks standard
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("employees")

print("✅ Delta table created!")

✅ Delta table created!


In [0]:
# Drop existing table first, then recreate
spark.sql("DROP TABLE IF EXISTS employees")

df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("employees")

print("✅ Delta table created!")

# Verify
spark.sql("SELECT * FROM employees").show()

✅ Delta table created!
+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



In [0]:
# Read it back
df_delta = spark.read.table("employees")
display(df_delta)
print("Rows read back:", df_delta.count())

name,dept,city,salary,gender,age,joining_date
Ravi,Engineering,Pune,55000,M,28,2021-03-15
Priya,HR,Mumbai,42000,F,32,2019-07-22
Arjun,Engineering,Delhi,72000,M,26,2022-01-10
Sneha,Finance,Pune,61000,F,30,2020-11-05
Rohit,Engineering,Mumbai,80000,M,35,2018-06-30
Meera,HR,Bangalore,39000,F,27,2023-02-18
Karan,Finance,Delhi,55000,M,29,2021-09-12
Divya,Engineering,Pune,91000,F,33,2017-04-25
Nitin,HR,Mumbai,44000,M,31,2020-03-08
Anjali,Finance,Bangalore,67000,F,28,2019-11-30


Rows read back: 10


In [0]:
from pyspark.sql.functions import col, to_date

# Convert date column
df = df.withColumn(
    "joining_date",
    to_date(col("joining_date"), "yyyy-MM-dd")
)

# Write partitioned by dept
df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("dept") \
  .saveAsTable("employees_partitioned")

print("✅ Partitioned Delta table created!")

✅ Partitioned Delta table created!


In [0]:
# Read only Engineering dept
df_eng = spark.sql("""
    SELECT * FROM employees_partitioned
    WHERE dept = 'Engineering'
""")
display(df_eng)

name,dept,city,salary,gender,age,joining_date
Ravi,Engineering,Pune,55000,M,28,2021-03-15
Arjun,Engineering,Delhi,72000,M,26,2022-01-10
Rohit,Engineering,Mumbai,80000,M,35,2018-06-30
Divya,Engineering,Pune,91000,F,33,2017-04-25


In [0]:
from pyspark.sql.functions import upper, when, col, to_date

# Clean + Transform
df_clean = df \
    .dropna(subset=["name", "salary"]) \
    .withColumn("name", upper(col("name"))) \
    .withColumn("salary_grade",
        when(col("salary") > 70000, "High")
        .when(col("salary") > 50000, "Medium")
        .otherwise("Low")) \
    .dropDuplicates(["name"])

# Write as Delta
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("employees_clean")

print("✅ Pipeline complete!")
print("Rows written:", df_clean.count())
display(df_clean)

✅ Pipeline complete!
Rows written: 10


name,dept,city,salary,gender,age,joining_date,salary_grade
RAVI,Engineering,Pune,55000,M,28,2021-03-15,Medium
PRIYA,HR,Mumbai,42000,F,32,2019-07-22,Low
ARJUN,Engineering,Delhi,72000,M,26,2022-01-10,High
SNEHA,Finance,Pune,61000,F,30,2020-11-05,Medium
ROHIT,Engineering,Mumbai,80000,M,35,2018-06-30,High
MEERA,HR,Bangalore,39000,F,27,2023-02-18,Low
KARAN,Finance,Delhi,55000,M,29,2021-09-12,Medium
DIVYA,Engineering,Pune,91000,F,33,2017-04-25,High
NITIN,HR,Mumbai,44000,M,31,2020-03-08,Low
ANJALI,Finance,Bangalore,67000,F,28,2019-11-30,Medium


In [0]:
df =df_clean

df.select("name", "salary_grade").show()


+------+------------+
|  name|salary_grade|
+------+------------+
|  RAVI|      Medium|
| PRIYA|         Low|
| ARJUN|        High|
| SNEHA|      Medium|
| ROHIT|        High|
| MEERA|         Low|
| KARAN|      Medium|
| DIVYA|        High|
| NITIN|         Low|
|ANJALI|      Medium|
+------+------------+



In [0]:
df = spark.table("employees")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df.select("name", "salary").show()

+------+------+
|  name|salary|
+------+------+
|  Ravi| 55000|
| Priya| 42000|
| Arjun| 72000|
| Sneha| 61000|
| Rohit| 80000|
| Meera| 39000|
| Karan| 55000|
| Divya| 91000|
| Nitin| 44000|
|Anjali| 67000|
+------+------+



In [0]:
df.filter(col("city") == "Pune").show()

+-----+-----------+----+------+------+---+------------+
| name|       dept|city|salary|gender|age|joining_date|
+-----+-----------+----+------+------+---+------------+
| Ravi|Engineering|Pune| 55000|     M| 28|  2021-03-15|
|Sneha|    Finance|Pune| 61000|     F| 30|  2020-11-05|
|Divya|Engineering|Pune| 91000|     F| 33|  2017-04-25|
+-----+-----------+----+------+------+---+------------+



In [0]:
df.filter(col("salary")> 60000).show()

+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



In [0]:
df.filter(col("gender")=="F").show()

+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



In [0]:
df.filter(col("age")>= 30).show()

+-----+-----------+------+------+------+---+------------+
| name|       dept|  city|salary|gender|age|joining_date|
+-----+-----------+------+------+------+---+------------+
|Priya|         HR|Mumbai| 42000|     F| 32|  2019-07-22|
|Sneha|    Finance|  Pune| 61000|     F| 30|  2020-11-05|
|Rohit|Engineering|Mumbai| 80000|     M| 35|  2018-06-30|
|Divya|Engineering|  Pune| 91000|     F| 33|  2017-04-25|
|Nitin|         HR|Mumbai| 44000|     M| 31|  2020-03-08|
+-----+-----------+------+------+------+---+------------+



In [0]:
df.select("name", "dept", "city").orderBy("name").show()

+------+-----------+---------+
|  name|       dept|     city|
+------+-----------+---------+
|Anjali|    Finance|Bangalore|
| Arjun|Engineering|    Delhi|
| Divya|Engineering|     Pune|
| Karan|    Finance|    Delhi|
| Meera|         HR|Bangalore|
| Nitin|         HR|   Mumbai|
| Priya|         HR|   Mumbai|
|  Ravi|Engineering|     Pune|
| Rohit|Engineering|   Mumbai|
| Sneha|    Finance|     Pune|
+------+-----------+---------+



In [0]:
df.count()

10

In [0]:
# df.groupBy("dept").filterBy("Engineering").show()
df.filter(
    (col("dept") == "Engineering") & (col("salary")>70000)
).show()


+-----+-----------+------+------+------+---+------------+
| name|       dept|  city|salary|gender|age|joining_date|
+-----+-----------+------+------+------+---+------------+
|Arjun|Engineering| Delhi| 72000|     M| 26|  2022-01-10|
|Rohit|Engineering|Mumbai| 80000|     M| 35|  2018-06-30|
|Divya|Engineering|  Pune| 91000|     F| 33|  2017-04-25|
+-----+-----------+------+------+------+---+------------+



In [0]:
# Show employees fromPune OR Mumbai

df.filter((col("city") == "Pune") | (col("city") == "Mumbai")).show()


+-----+-----------+------+------+------+---+------------+
| name|       dept|  city|salary|gender|age|joining_date|
+-----+-----------+------+------+------+---+------------+
| Ravi|Engineering|  Pune| 55000|     M| 28|  2021-03-15|
|Priya|         HR|Mumbai| 42000|     F| 32|  2019-07-22|
|Sneha|    Finance|  Pune| 61000|     F| 30|  2020-11-05|
|Rohit|Engineering|Mumbai| 80000|     M| 35|  2018-06-30|
|Divya|Engineering|  Pune| 91000|     F| 33|  2017-04-25|
|Nitin|         HR|Mumbai| 44000|     M| 31|  2020-03-08|
+-----+-----------+------+------+------+---+------------+



In [0]:
# Show employees whose age is between 28 and 32.
df.filter((col("age") >= 28) & (col("age") <= 32)).show()


+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



In [0]:
# Show employees whose name starts with
df.filter(col("name").startswith("A")).show()


+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



In [0]:
# Show employees whose department is NOT HR.
df.filter(col("dept") != "HR").show()

+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



In [0]:
df = spark.table("employees")
df.show()

+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



In [0]:
df.withColumn("bonus", "salary" *0.1).show()

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-5128642686430693>, line 1
----> 1 df.withColumn("bonus", "salary" *0.1).show()

TypeError: can't multiply sequence by non-int of type 'float'

In [0]:
from pyspark.sql.functions import col

df.withColumn(
    "bonus",
    col("salary") * 0.10
).show()

+------+-----------+---------+------+------+---+------------+------+
|  name|       dept|     city|salary|gender|age|joining_date| bonus|
+------+-----------+---------+------+------+---+------------+------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|5500.0|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|4200.0|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|7200.0|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|6100.0|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|8000.0|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|3900.0|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|5500.0|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|9100.0|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|4400.0|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|6700.0|
+------+-----------+---------+------+------+---+------------+------+



In [0]:
df.withColumn("bonus", col("salary") *0.1).show()

+------+-----------+---------+------+------+---+------------+------+
|  name|       dept|     city|salary|gender|age|joining_date| bonus|
+------+-----------+---------+------+------+---+------------+------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|5500.0|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|4200.0|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|7200.0|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|6100.0|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|8000.0|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|3900.0|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|5500.0|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|9100.0|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|4400.0|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|6700.0|
+------+-----------+---------+------+------+---+------------+------+



In [0]:
df.withColumn("annual salary", col("salary") *12 ).show()

+------+-----------+---------+------+------+---+------------+-------------+
|  name|       dept|     city|salary|gender|age|joining_date|annual salary|
+------+-----------+---------+------+------+---+------------+-------------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|       660000|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|       504000|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|       864000|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|       732000|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|       960000|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|       468000|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|       660000|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|      1092000|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|       528000|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|       804000|
+------+----

In [0]:
from pyspark.sql.functions import *

df.withColumn(
    "name",
    lower(col("name"))
).show()

+------+-----------+---------+------+------+---+------------+
|  name|       dept|     city|salary|gender|age|joining_date|
+------+-----------+---------+------+------+---+------------+
|  ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|
| priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|
| arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|
| sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|
| rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|
| meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|
| karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|
| divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|
| nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|
|anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|
+------+-----------+---------+------+------+---+------------+



# Create a column

tax
salary >70000

20%

otherwise

10%




In [0]:
df.withColumn("tax", when (col("salary") > 70000, 0.2 * col("salary")).otherwise ( 0.1 * col("salary"))).show()


+------+-----------+---------+------+------+---+------------+-------+
|  name|       dept|     city|salary|gender|age|joining_date|    tax|
+------+-----------+---------+------+------+---+------------+-------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15| 5500.0|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22| 4200.0|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|14400.0|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05| 6100.0|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|16000.0|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18| 3900.0|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12| 5500.0|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|18200.0|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08| 4400.0|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30| 6700.0|
+------+-----------+---------+------+------+---+------------+-------+



In [0]:
 
# Create Experience Level Age >32 Senior Age 28-32 Mid  Else Junior
from pyspark.sql.functions import *
df = spark.table("employees")
df.withColumn("experience_level", when (col("age") > 32, "Senior").when(col("age").between(28,32), "Mid").otherwise("Junior")).show()


+------+-----------+---------+------+------+---+------------+----------------+
|  name|       dept|     city|salary|gender|age|joining_date|experience_level|
+------+-----------+---------+------+------+---+------------+----------------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|             Mid|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|             Mid|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|          Junior|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|             Mid|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|          Senior|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|          Junior|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|             Mid|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|          Senior|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|             Mid|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  20

In [0]:
df.withColumn(
    "location",
    when(col("city") == "Pune" , "west")
    .when ( col("city") == "Mumbai", "west")
    .when(col("city") == "Bangalore", "south")
    .when(col("city") == "Delhi", "north")
    ).show()

+------+-----------+---------+------+------+---+------------+--------+
|  name|       dept|     city|salary|gender|age|joining_date|location|
+------+-----------+---------+------+------+---+------------+--------+
|  Ravi|Engineering|     Pune| 55000|     M| 28|  2021-03-15|    west|
| Priya|         HR|   Mumbai| 42000|     F| 32|  2019-07-22|    west|
| Arjun|Engineering|    Delhi| 72000|     M| 26|  2022-01-10|   north|
| Sneha|    Finance|     Pune| 61000|     F| 30|  2020-11-05|    west|
| Rohit|Engineering|   Mumbai| 80000|     M| 35|  2018-06-30|    west|
| Meera|         HR|Bangalore| 39000|     F| 27|  2023-02-18|   south|
| Karan|    Finance|    Delhi| 55000|     M| 29|  2021-09-12|   north|
| Divya|Engineering|     Pune| 91000|     F| 33|  2017-04-25|    west|
| Nitin|         HR|   Mumbai| 44000|     M| 31|  2020-03-08|    west|
|Anjali|    Finance|Bangalore| 67000|     F| 28|  2019-11-30|   south|
+------+-----------+---------+------+------+---+------------+--------+



In [0]:
df.groupBy("dept").agg(avg("salary")).show()


+-----------+------------------+
|       dept|       avg(salary)|
+-----------+------------------+
|Engineering|           74500.0|
|         HR|41666.666666666664|
|    Finance|           61000.0|
+-----------+------------------+



In [0]:
df.select(avg("salary")).show()

+-----------+
|avg(salary)|
+-----------+
|    60600.0|
+-----------+



In [0]:
df.select(max("salary")).show()

+-----------+
|max(salary)|
+-----------+
|      91000|
+-----------+



In [0]:
df.select(min("salary")).show()

+-----------+
|min(salary)|
+-----------+
|      39000|
+-----------+



In [0]:
df.select(sum("salary")).alias ("Total salary").show()

+-----------+
|sum(salary)|
+-----------+
|     606000|
+-----------+

